In [1]:
# =============================================
#                   IMPORT
# =============================================
from keras.optimizers                   import Adam
from keras.callbacks                    import EarlyStopping, ReduceLROnPlateau
from preProcessingData                  import PreProcessData
# from model.systemModel                  import SystemModel as systemModel
from model.componentModel               import SystemModel as componentModel
from sklearn.preprocessing              import MinMaxScaler

from tqdm                               import tqdm
from datetime                           import datetime
from prettytable                        import PrettyTable
import plotly.graph_objects             as go
import tensorflow                       as tf
import numpy                            as np 
import pandas                           as pd
import matplotlib.pyplot                as plt
import plotly.express                   as px

import os, warnings
import random
import json

# ====================================
# Set debug level and 
# ------------------------------------
os.environ["CUDA_VISIBLE_DEVICES"] = "0,1"  
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '1'  # or any {'0', '1', '2'}
warnings.filterwarnings('ignore') 

np.random.seed(1)
random.seed(1)
tf.random.set_seed(1)

# ====================================
# Config
# ------------------------------------
with open('utility/modelConfig.json', 'r') as configFile:
    config = json.load(configFile)
myConfig   = config["system"]

2024-11-29 18:19:35.394861: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  AVX2 AVX_VNNI FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.
2024-11-29 18:19:35.465684: I tensorflow/core/util/util.cc:169] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2024-11-29 18:19:35.485911: E tensorflow/stream_executor/cuda/cuda_blas.cc:2981] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2024-11-29 18:19:35.901815: W tensorflow/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libnvinfer.so.7'; dlerror: li

In [2]:
preData                 = PreProcessData(config = myConfig)                    # Preprocessing data
data                    = preData.loadFile( dir     = "data/", 
                                            name    = "HIDataWithTrajectory",
                                            type    = "xlsx")

Process:227416||2024-11-29 18:19:37,435-INFO [preProcessingData.py:loadFile():592]:Loading the xlsx file at the directory data/HIDataWithTrajectory ...
Process:227416||2024-11-29 18:20:07,343-INFO [preProcessingData.py:loadFile():598]:Finish loading the xlsx file at the directory data/HIDataWithTrajectory.


Index(['Time', 'norHICr1', 'norHICr2', 'norHICv2', 'System_HI',
       'OriginalHICr1', 'OriginalHICr2', 'OriginalHICv2', 'trajectoryCr1',
       'trajectoryCr2', 'trajectoryCv2', 'trajectorySys'],
      dtype='object')


In [3]:
# typeOfPadding = "meanPrevious"
validColumn = ["OriginalHICr1", "OriginalHICr2", "OriginalHICv2"]
trajectory  = ["trajectoryCr1", "trajectoryCr2", "trajectoryCv2", "trajectorySys"]

dfFiltered  = data.copy()

# if typeOfPadding == "linear":
#     dfFiltered[validColumn] = dfFiltered[validColumn].interpolate(method='linear')

# elif typeOfPadding == "zero":
#     dfFiltered[validColumn] = dfFiltered[validColumn].fillna(0)

# elif typeOfPadding == "spline":
#     dfFiltered[validColumn] = dfFiltered[validColumn].interpolate(method='spline', order=2)

# elif typeOfPadding == "meanPrevious":
#     for col in validColumn:
#         # print(dfFiltered[col])
#         rolling_mean    = dfFiltered[col].rolling(window=180, min_periods=1, closed='left').mean() 
#         dfFiltered[col] = dfFiltered[col].fillna(rolling_mean)
#     dfFiltered[validColumn] = dfFiltered[validColumn].interpolate(method='linear')
# else:
#     dfFiltered = dfFiltered

# dfFiltered[validColumn] = dfFiltered[validColumn].fillna(method='ffill')
# dfFiltered[validColumn] = dfFiltered[validColumn].fillna(method='bfill')

In [4]:
# smothTime = 1
# for idx in range(smothTime):
validColumn = ["OriginalHICr1", "OriginalHICr2", "OriginalHICv2"]
trajectory  = ["trajectoryCr1", "trajectoryCr2", "trajectoryCv2", "trajectorySys"]
for idx,col in enumerate(validColumn):
    traject     = trajectory[idx]
    max_traject = dfFiltered[traject].max()

    for idxTraject in range(max_traject):
        idxTraject                  += 1
        mask                        = dfFiltered[traject] == idxTraject
        dfFiltered.loc[mask, col]   = (dfFiltered.loc[mask, col].rolling(window=200, min_periods=1, closed='left').mean())



In [5]:
scaler = MinMaxScaler()
dfFiltered[validColumn] = scaler.fit_transform(dfFiltered[validColumn])
dfFiltered["System_HI"] = dfFiltered[['OriginalHICr1', 'OriginalHICr2', 'OriginalHICv2']].min(axis=1)

In [7]:
# Filter data for a specific time range (e.g., time between 4 and 10)
time_filtered_df = dfFiltered[(dfFiltered['Time'] >= 1921) & (dfFiltered['Time'] <= 3053)]

plotColumn = ["OriginalHICr1", "OriginalHICr2", "OriginalHICv2","System_HI"]
trajectory  = ["trajectoryCr1", "trajectoryCr2", "trajectoryCv2", "trajectorySys"]

# Plotting with Plotly
fig1 = {}

for idx, signal in enumerate(plotColumn):
    fig1[idx] = px.line(
        time_filtered_df,
        x='Time',
        y=signal,

        color=trajectory[idx],

        title='Value1 per Time for Different IDs',

        labels={'time': 'Time', 'value1': 'Value1', 'id': 'ID'}

    )



    fig1[idx].update_layout(

        xaxis_title='Time',

        yaxis_title='OriginalHICr1',

        legend_title='ID',

        template='plotly_white'

    )



    fig1[idx].show()

In [ ]:
import plotly.express as px

plotColumn = ["OriginalHICr1", "OriginalHICr2", "OriginalHICv2","System_HI"]
trajectory  = ["trajectoryCr1", "trajectoryCr2", "trajectoryCv2", "trajectorySys"]

# Plotting with Plotly
fig = {}

for idx, signal in enumerate(plotColumn):
    fig[idx] = px.line(
        dfFiltered,
        x='Time',
        y=signal,

        color=trajectory[idx],

        title='Value1 per Time for Different IDs',

        labels={'time': 'Time', 'value1': 'Value1', 'id': 'ID'}

    )



    fig[idx].update_layout(

        xaxis_title='Time',

        yaxis_title='OriginalHICr1',

        legend_title='ID',

        template='plotly_white'

    )



    fig[idx].show()

In [7]:

# fig1 = {}
# signals = ["norHICr1", "norHICr2", "norHICv2", "System_HI"]
# trajectory  = ["trajectoryCr1", "trajectoryCr2", "trajectoryCv2", "trajectorySys"]


# for idx, signal in enumerate(signals):
#     fig1[idx] = px.line(
#         data,
#         x='Time',
#         y=signal,

#         color=trajectory[idx],

#         title='Value1 per Time for Different IDs',

#         labels={'time': 'Time', 'value1': 'Value1', 'id': 'ID'}

#     )

#     fig1[idx].update_layout(

#         xaxis_title='Time',

#         yaxis_title='OriginalHICr1',

#         legend_title='ID',

#         template='plotly_white'
#     )

#     fig1[idx].show()